# 02 — Research Pipeline

        **Mode:** `live`  
        **Version:** Praval `0.8.0`  
        **Course link:** [Watch the original course video](https://www.youtube.com/watch?v=u7gib_vrMMQ)

        This notebook makes the framework's execution visible. It records actual
        stages and renders the messages or runtime events that connect them.

        **You will see**

        - A real provider call enriches a source note.
- A second agent turns findings into a concise briefing.
- The Reef preserves the intermediate model output as a Spore.

        Run the cells from top to bottom. Committed notebooks contain no saved
        output, so everything shown is produced by your installed Praval package.


In [ ]:
from __future__ import annotations

import html as _html
import json
import os
import time
from pathlib import Path

from IPython.display import HTML, display

os.environ.setdefault("PRAVAL_OBSERVABILITY", "off")


from praval import get_provider_registry
from praval.models import ModelResponse, ProviderCapabilities


class NotebookLifecycleProvider:
    """Credential-free adapter for agents whose handlers do not call a model."""

    provider_name = "notebook-lifecycle"
    capabilities = ProviderCapabilities(text=True)

    def __init__(self, config):
        self.config = config

    def invoke(self, request, tools=None):
        return ModelResponse(
            content="Notebook lifecycle response",
            provider=self.provider_name,
            model=request.model,
        )

    def close(self):
        return None


_notebook_registry = get_provider_registry()
_notebook_registry.register_provider(
    "notebook-lifecycle",
    NotebookLifecycleProvider,
    default_model="notebook-lifecycle-model",
    capabilities=NotebookLifecycleProvider.capabilities,
)
os.environ.setdefault("PRAVAL_DEFAULT_PROVIDER", "notebook-lifecycle")
os.environ.setdefault("PRAVAL_DEFAULT_MODEL", "notebook-lifecycle-model")

EVENTS = []
_STARTED = time.perf_counter()


def stage(actor, action, detail="", spore=None):
    """Capture one observable execution stage."""
    EVENTS.append(
        {
            "ms": round((time.perf_counter() - _STARTED) * 1000, 1),
            "actor": actor,
            "action": action,
            "detail": detail,
            "spore_id": getattr(spore, "id", "") if spore else "",
        }
    )


def show_flow(*steps):
    cards = []
    for index, (name, detail) in enumerate(steps):
        if index:
            cards.append('<div style="font-size:24px;color:#557">→</div>')
        cards.append(
            '<div style="padding:12px 16px;border:1px solid #b8c7dc;'
            'border-radius:12px;background:#f7fbff;min-width:130px">'
            f'<b>{_html.escape(name)}</b><br>'
            f'<span style="color:#456;font-size:12px">'
            f'{_html.escape(detail)}</span></div>'
        )
    display(
        HTML(
            '<div style="display:flex;gap:10px;align-items:center;'
            'flex-wrap:wrap;margin:12px 0">' + "".join(cards) + "</div>"
        )
    )


def show_timeline(events=None):
    events = EVENTS if events is None else events
    rows = []
    for item in events:
        rows.append(
            "<tr>"
            f"<td>{item['ms']:.1f}</td>"
            f"<td><b>{_html.escape(str(item['actor']))}</b></td>"
            f"<td>{_html.escape(str(item['action']))}</td>"
            f"<td>{_html.escape(str(item['detail']))}</td>"
            f"<td><code>{_html.escape(str(item['spore_id'])[:12])}</code></td>"
            "</tr>"
        )
    display(
        HTML(
            '<table style="border-collapse:collapse;width:100%">'
            '<thead><tr><th>ms</th><th>actor</th><th>stage</th>'
            '<th>detail</th><th>spore</th></tr></thead><tbody>'
            + "".join(rows)
            + "</tbody></table>"
        )
    )


def show_spore(spore, label="Spore on the Reef"):
    payload = json.loads(spore.to_json())
    rendered = _html.escape(json.dumps(payload, indent=2, sort_keys=True))
    display(
        HTML(
            '<div style="border-left:5px solid #7b61ff;padding:10px 14px;'
            'background:#faf9ff;border-radius:8px">'
            f'<b>{_html.escape(label)}</b><pre style="white-space:pre-wrap">'
            f"{rendered}</pre></div>"
        )
    )


def show_json(value, label="Result"):
    rendered = _html.escape(json.dumps(value, indent=2, sort_keys=True, default=str))
    display(
        HTML(
            '<div style="border-left:5px solid #18a999;padding:10px 14px;'
            'background:#f5fffd;border-radius:8px">'
            f'<b>{_html.escape(label)}</b><pre style="white-space:pre-wrap">'
            f"{rendered}</pre></div>"
        )
    )


def require_env(*names):
    missing = [name for name in names if not os.environ.get(name)]
    if missing:
        raise RuntimeError("Missing required notebook configuration: " + ", ".join(missing))
    return {name: os.environ[name] for name in names}


def find_example_asset(relative):
    relative = Path(relative)
    starts = [Path.cwd(), *Path.cwd().parents]
    for root in starts:
        for candidate in (root / relative, root / "examples" / relative):
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f"Could not locate example asset: {relative}")


## Live prerequisites

By default this notebook uses OpenAI. Set `PRAVAL_TUTORIAL_PROVIDER` to
`anthropic` to use Anthropic instead, along with the matching key and model
variable. Missing configuration is an error, not a skipped demonstration.


In [ ]:
provider = os.environ.get("PRAVAL_TUTORIAL_PROVIDER", "openai").lower()
configuration = {
    "openai": ("OPENAI_API_KEY", "PRAVAL_OPENAI_MODEL"),
    "anthropic": ("ANTHROPIC_API_KEY", "PRAVAL_ANTHROPIC_MODEL"),
}
if provider not in configuration:
    raise RuntimeError("PRAVAL_TUTORIAL_PROVIDER must be openai or anthropic")
key_name, model_name = configuration[provider]
values = require_env(key_name, model_name)
model = values[model_name]
show_json({"provider": provider, "model": model}, "Live model selection")


In [ ]:
from praval import agent, broadcast, chat, get_reef, start_agents
from praval.core.reef import reset_reef

reset_reef()
briefings = []
intermediate_spores = []
source_note = (
    "Praval agents communicate through structured Spores on a Reef. "
    "Handlers filter messages by type, and completion waits include "
    "cascading work triggered by later broadcasts."
)


@agent(
    "course-researcher",
    provider=provider,
    model=model,
    responds_to=["research_request"],
    auto_broadcast=False,
    config={"temperature": 0, "max_output_tokens": 160},
)
def researcher(spore):
    stage("researcher", "model request", "extract three findings", spore)
    findings = chat(
        "Extract exactly three short factual findings from this source:\n"
        + spore.knowledge["source"]
    )
    broadcast({"type": "findings", "text": findings})
    stage("researcher", "broadcast", "findings", spore)


@agent(
    "course-writer",
    provider=provider,
    model=model,
    responds_to=["findings"],
    auto_broadcast=False,
    config={"temperature": 0, "max_output_tokens": 120},
)
def writer(spore):
    intermediate_spores.append(spore)
    stage("writer", "model request", "compose briefing", spore)
    briefing = chat(
        "Turn these findings into a two-sentence technical briefing:\n"
        + spore.knowledge["text"]
    )
    briefings.append(briefing)
    stage("writer", "briefing complete", f"{len(briefing)} chars", spore)


In [ ]:
start_agents(
    researcher,
    writer,
    initial_data={"type": "research_request", "source": source_note},
    channel="course-research",
)
reef = get_reef()
assert reef.wait_for_completion(timeout=180)
assert len(intermediate_spores) == 1
assert len(briefings) == 1 and len(briefings[0].strip()) > 20


In [ ]:
show_spore(intermediate_spores[0], "Model findings passed to the writer")
show_json({"briefing": briefings[0]}, "Final live briefing")
show_timeline()


In [ ]:
researcher._praval_agent.close()
writer._praval_agent.close()
assert reef.shutdown(timeout=10)
stage("reef", "shutdown", "live agents and channels closed")
